In [0]:
# Set up direct access credential
spark.conf.set(
    "fs.azure.account.key.storageppg.dfs.core.windows.net",
    "<secret-key>"
)

print("Storage account connected ✅")

Storage account connected ✅


In [0]:
from pyspark.sql import functions as F

materials         = spark.read.option("header", True).csv("abfss://bronze@storageppg.dfs.core.windows.net/ingested/materials.csv")
inventory         = spark.read.option("header", True).csv("abfss://bronze@storageppg.dfs.core.windows.net/ingested/inventory_snapshot.csv")
purchase_orders   = spark.read.option("header", True).csv("abfss://bronze@storageppg.dfs.core.windows.net/ingested/purchase_orders.csv")
production_orders = spark.read.option("header", True).csv("abfss://bronze@storageppg.dfs.core.windows.net/ingested/production_orders.csv")
suppliers         = spark.read.option("header", True).csv("abfss://bronze@storageppg.dfs.core.windows.net/ingested/suppliers.csv")
sales_orders      = spark.read.option("header", True).csv("abfss://bronze@storageppg.dfs.core.windows.net/ingested/sales_orders.csv")

print("All 6 files loaded ✅")

All 6 files loaded ✅


In [0]:
def quality_check(df, name, key_cols, date_cols=[], numeric_cols=[]):
    print(f"\n========== {name} ==========")
    total = df.count()
    print(f"Total rows: {total}")

    # Null check on key columns
    for col in key_cols:
        null_count = df.filter(F.col(col).isNull() | (F.col(col) == "")).count()
        print(f"  Nulls in '{col}': {null_count}")

    # Duplicate check
    dup_count = total - df.dropDuplicates().count()
    print(f"  Duplicate rows: {dup_count}")

    # Cast date columns
    for col in date_cols:
        df = df.withColumn(col, F.to_date(F.col(col)))
        invalid = df.filter(F.col(col).isNull()).count()
        print(f"  Invalid dates in '{col}': {invalid}")

    # Cast numeric columns
    for col in numeric_cols:
        df = df.withColumn(col, F.col(col).cast("double"))
        invalid = df.filter(F.col(col).isNull()).count()
        print(f"  Invalid numerics in '{col}': {invalid}")

    # Drop duplicates
    df = df.dropDuplicates()
    print(f"  Rows after dedup: {df.count()}")

    return df

In [0]:
materials_clean = quality_check(
    materials, "materials",
    key_cols=["material_id", "material_name", "category"],
    numeric_cols=["reorder_level", "lead_time_days", "unit_cost"]
)

inventory_clean = quality_check(
    inventory, "inventory_snapshot",
    key_cols=["material_id", "snapshot_date", "quantity_on_hand"],
    date_cols=["snapshot_date", "lot_expiry_date"],
    numeric_cols=["quantity_on_hand"]
)

purchase_clean = quality_check(
    purchase_orders, "purchase_orders",
    key_cols=["po_id", "material_id", "supplier_id"],
    date_cols=["po_date", "expected_delivery_date", "actual_delivery_date"],
    numeric_cols=["quantity_ordered", "quantity_received"]
)

production_clean = quality_check(
    production_orders, "production_orders",
    key_cols=["production_order_id", "material_id", "finished_product"],
    date_cols=["production_date"],
    numeric_cols=["quantity_consumed"]
)

suppliers_clean = quality_check(
    suppliers, "suppliers",
    key_cols=["supplier_id", "supplier_name", "country", "reliability_rating"]
)

sales_clean = quality_check(
    sales_orders, "sales_orders",
    key_cols=["sales_order_id", "customer_name", "finished_product"],
    date_cols=["order_date", "required_delivery_date"],
    numeric_cols=["quantity_ordered"]
)


========== materials ==========
Total rows: 20
  Nulls in 'material_id': 0
  Nulls in 'material_name': 0
  Nulls in 'category': 0
  Duplicate rows: 0
  Invalid numerics in 'reorder_level': 0
  Invalid numerics in 'lead_time_days': 0
  Invalid numerics in 'unit_cost': 0
  Rows after dedup: 20

========== inventory_snapshot ==========
Total rows: 240
  Nulls in 'material_id': 0
  Nulls in 'snapshot_date': 0
  Nulls in 'quantity_on_hand': 0
  Duplicate rows: 0
  Invalid dates in 'snapshot_date': 0
  Invalid dates in 'lot_expiry_date': 96
  Invalid numerics in 'quantity_on_hand': 0
  Rows after dedup: 240

========== purchase_orders ==========
Total rows: 40
  Nulls in 'po_id': 0
  Nulls in 'material_id': 0
  Nulls in 'supplier_id': 0
  Duplicate rows: 0
  Invalid dates in 'po_date': 0
  Invalid dates in 'expected_delivery_date': 0
  Invalid dates in 'actual_delivery_date': 1
  Invalid numerics in 'quantity_ordered': 0
  Invalid numerics in 'quantity_received': 0
  Rows after dedup: 40

=

In [0]:
base_out = "abfss://bronze@storageppg.dfs.core.windows.net/validated"

materials_clean.write.mode("overwrite").option("header", True).csv(f"{base_out}/materials")
inventory_clean.write.mode("overwrite").option("header", True).csv(f"{base_out}/inventory_snapshot")
purchase_clean.write.mode("overwrite").option("header", True).csv(f"{base_out}/purchase_orders")
production_clean.write.mode("overwrite").option("header", True).csv(f"{base_out}/production_orders")
suppliers_clean.write.mode("overwrite").option("header", True).csv(f"{base_out}/suppliers")
sales_clean.write.mode("overwrite").option("header", True).csv(f"{base_out}/sales_orders")

print("All validated files written to bronze/validated ✅")

All validated files written to bronze/validated ✅


In [0]:
for folder in ["materials", "inventory_snapshot", "purchase_orders",
               "production_orders", "suppliers", "sales_orders"]:
    files = dbutils.fs.ls(f"abfss://bronze@storageppg.dfs.core.windows.net/validated/{folder}")
    print(f"{folder}: {len(files)} file(s) ✅")

materials: 4 file(s) ✅
inventory_snapshot: 4 file(s) ✅
purchase_orders: 4 file(s) ✅
production_orders: 4 file(s) ✅
suppliers: 4 file(s) ✅
sales_orders: 4 file(s) ✅
